<a href="https://colab.research.google.com/github/harshithadondeti/self-rag-factual-question-answering/blob/main/LLM_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install datasets transformers sentence-transformers faiss-cpu accelerate pandas matplotlib scikit-learn tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 86.3 MB/s eta 0:00:00


# imports & setup

In [ ]:
import gc
import os
import re
import random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import faiss

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)
pd.set_option("display.max_colwidth", 120)
plt.rcParams["figure.figsize"] = (10, 4)

# Dataset – SQuAD

In [ ]:
full_ds = load_dataset("squad", split="validation").shuffle(seed=SEED).select(range(120))
eval_ds = full_ds.select(range(20))

print(full_ds)
print(eval_ds)

corpus_texts = [ex["context"] for ex in full_ds]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 120
})
Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 20
})


# Retriever setup

In [ ]:
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

corpus_embeddings = embedder.encode(
    corpus_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    batch_size=16,
    show_progress_bar=True
).astype("float32")

index = faiss.IndexFlatIP(corpus_embeddings.shape[1])
index.add(corpus_embeddings)

def retrieve(question, top_k=3):
    q_emb = embedder.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")
    scores, idx = index.search(q_emb, top_k)
    passages = [corpus_texts[i] for i in idx[0]]
    return passages, scores[0].tolist()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

... (4 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

# Metrics & normalization

In [ ]:
def normalize_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def exact_match(pred: str, gold: str) -> float:
    return float(normalize_text(pred) == normalize_text(gold))

def token_f1(pred: str, gold: str) -> float:
    pred_tokens = normalize_text(pred).split()
    gold_tokens = normalize_text(gold).split()
    if not pred_tokens and not gold_tokens:
        return 1.0
    if not pred_tokens or not gold_tokens:
        return 0.0

    pred_counts = Counter(pred_tokens)
    gold_counts = Counter(gold_tokens)
    common = sum((pred_counts & gold_counts).values())
    if common == 0:
        return 0.0

    precision = common / len(pred_tokens)
    recall = common / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)

def best_over_answers(pred: str, answers: list[str], fn) -> float:
    return max(fn(pred, ans) for ans in answers) if answers else 0.0

def support_score(answer: str, passages: list[str]) -> float:
    # lightweight "groundedness" proxy
    context = normalize_text(" ".join(passages))
    answer_tokens = [
        t for t in normalize_text(answer).split()
        if t not in ENGLISH_STOP_WORDS and len(t) > 2
    ]
    if not answer_tokens:
        return 0.0
    answer_tokens = list(dict.fromkeys(answer_tokens))  # unique, keep order
    hits = sum(1 for t in answer_tokens if t in context)
    return hits / len(answer_tokens)

# Model loading, prompt & Self-RAG-lite

In [ ]:
MODEL_CHECKPOINTS = {
    "LaMini-Flan-T5-77M": "MBZUAI/LaMini-Flan-T5-77M",
    "Flan-T5-Small": "google/flan-t5-small",
    "Flan-T5-Base": "google/flan-t5-base",
}

def load_seq2seq(checkpoint: str):
    tokenizer = AutoTokenizer.from_pretrained(checkpoint, use_fast=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(
        checkpoint,
        low_cpu_mem_usage=True
    )
    model.eval()
    return tokenizer, model

def generate_text(tokenizer, model, prompt: str, max_new_tokens: int = 64) -> str:
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=2,
            do_sample=False,
            early_stopping=True
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()

def build_prompt(question: str, passages=None, mode="no_rag"):
    if mode == "no_rag":
        return f"Question: {question}\nAnswer briefly:"
    context = "\n\n".join([f"[{i+1}] {p}" for i, p in enumerate(passages)])
    if mode == "rag":
        return f"Passages:\n{context}\n\nQuestion: {question}\nAnswer using only the passages:"
    return ""

def self_rag_lite(tokenizer, model, question: str, passages: list[str]):
    is_year_query = any(word in question.lower() for word in ["year", "when", "date"])
    # Include more specific keywords for better anchoring
    keywords = [w for w in re.sub(r'[^a-z ]', '', question.lower()).split() if len(w) > 3 and w not in ENGLISH_STOP_WORDS]

    for p in passages[:5]: # Search deeper
        sentences = [s.strip() for s in p.split('.') if len(s.strip()) > 10]

        # Rank sentences by keyword density
        scored_sentences = []
        for s in sentences:
            score = sum(1 for k in keywords if k in s.lower())
            scored_sentences.append((score, s))

        # Process sentences with at least one keyword match
        for score, evidence in sorted(scored_sentences, key=lambda x: x[0], reverse=True):
            if score == 0: continue

            if is_year_query and not any(c.isdigit() for c in evidence):
                continue

            extract_prompt = f"Context: {evidence}\nQuestion: {question}\nShort Answer:"
            final_answer = generate_text(tokenizer, model, extract_prompt, max_new_tokens=10)

            # STRICT GROUNDING: The answer MUST be a substring of the evidence
            if len(final_answer) > 0 and "don't know" not in final_answer.lower():
                if final_answer.lower() in evidence.lower():
                    return final_answer, evidence

    return "Not found", "No relevant evidence identified in search"

# evaluate_model_on_rag

In [ ]:
def evaluate_model_on_rag(checkpoint: str, model_name: str, top_k: int = 3):
    tokenizer, model = load_seq2seq(checkpoint)

    rows = []
    for ex_id, ex in enumerate(tqdm(eval_ds, desc=f"Evaluating {model_name}")):
        question = ex["question"]
        gold_answers = ex["answers"]["text"]

        passages, scores = retrieve(question, top_k=top_k)
        prompt = build_prompt(question, passages, mode="rag")
        pred = generate_text(tokenizer, model, prompt)

        em = best_over_answers(pred, gold_answers, exact_match)
        f1 = best_over_answers(pred, gold_answers, token_f1)
        support = support_score(pred, passages)
        retrieval_hit = float(
            any(normalize_text(ans) in normalize_text(" ".join(passages)) for ans in gold_answers)
        )

        rows.append({
            "example_id": ex_id,
            "model": model_name,
            "question": question,
            "gold": gold_answers[0],
            "prediction": pred,
            "em": em,
            "f1": f1,
            "support": support,
            "retrieval_hit": retrieval_hit,
            "top_passage": passages[0],
            "top_score": scores[0],
        })

    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return pd.DataFrame(rows)

# Model comparison loop

In [ ]:
all_model_results = []

for label, checkpoint in MODEL_CHECKPOINTS.items():
    df = evaluate_model_on_rag(checkpoint, label, top_k=3)
    all_model_results.append(df)

model_results = pd.concat(all_model_results, ignore_index=True)
model_summary = model_results.groupby("model", as_index=False)[["em", "f1", "support", "retrieval_hit"]].mean()

model_summary.sort_values("f1", ascending=False)

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Evaluating LaMini-Flan-T5-77M:   0%|          | 0/20 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Evaluating Flan-T5-Small:   0%|          | 0/20 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Evaluating Flan-T5-Base:   0%|          | 0/20 [00:00<?, ?it/s]

,model,em,f1,support,retrieval_hit
1,Flan-T5-Small,0.10,0.187955,0.650000,0.85
0,Flan-T5-Base,0.05,0.148938,0.850000,0.85
2,LaMini-Flan-T5-77M,0.05,0.132890,0.901116,0.85


# Best model and method comparison

In [ ]:
best_model_name = model_summary.sort_values("f1", ascending=False).iloc[0]["model"]
best_checkpoint = MODEL_CHECKPOINTS[best_model_name]

print("Best model from the RAG comparison:", best_model_name)

tokenizer, model = load_seq2seq(best_checkpoint)

method_rows = []
for ex_id, ex in enumerate(tqdm(eval_ds, desc=f"Method comparison on {best_model_name}")):
    question = ex["question"]
    gold_answers = ex["answers"]["text"]
    passages, _ = retrieve(question, top_k=3)

    # No RAG
    pred_no = generate_text(tokenizer, model, build_prompt(question, None, mode="no_rag"))
    # RAG
    pred_rag = generate_text(tokenizer, model, build_prompt(question, passages, mode="rag"))
    # Self-RAG-lite
    pred_self, draft_self = self_rag_lite(tokenizer, model, question, passages)

    for mode, pred in [
        ("no_rag", pred_no),
        ("rag", pred_rag),
        ("self_rag_lite", pred_self),
    ]:
        method_rows.append({
            "example_id": ex_id,
            "mode": mode,
            "question": question,
            "gold": gold_answers[0],
            "prediction": pred,
            "em": best_over_answers(pred, gold_answers, exact_match),
            "f1": best_over_answers(pred, gold_answers, token_f1),
            "support": support_score(pred, passages),
        })

del model, tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

method_results = pd.DataFrame(method_rows)
method_summary = method_results.groupby("mode", as_index=False)[["em", "f1", "support"]].mean()
method_summary.sort_values("f1", ascending=False)

Best model from the RAG comparison: Flan-T5-Small


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Method comparison on Flan-T5-Small:   0%|          | 0/20 [00:00<?, ?it/s]

,mode,em,f1,support
2,self_rag_lite,0.6,0.684524,0.75
1,rag,0.1,0.187955,0.65
0,no_rag,0.0,0.035000,0.10


In [ ]:
for i in range(3):
    ex = eval_ds[i]
    q = ex["question"]
    gold = ex["answers"]["text"][0]
    passages, _ = retrieve(q, top_k=5)
    tokenizer, model = load_seq2seq(best_checkpoint)

    print("="*100)
    print("QUESTION:", q)
    print("GOLD:", gold)

    pred_no = generate_text(tokenizer, model, build_prompt(q, None, "no_rag"))
    pred_self, evidence = self_rag_lite(tokenizer, model, q, passages)

    print("NO-RAG:", pred_no)
    print("EVIDENCE FOUND:", evidence)
    print("SELF-RAG FINAL ANSWER:", pred_self)
    print("="*100)

    del model, tokenizer
    gc.collect()

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


QUESTION: In what year did Massachusetts first require children to be educated in schools?
GOLD: 1852
NO-RAG: 1904
EVIDENCE FOUND: Private schooling in the United States has been debated by educators, lawmakers and parents, since the beginnings of compulsory education in Massachusetts in 1852
SELF-RAG FINAL ANSWER: 1852


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


QUESTION: When were stromules discovered?
GOLD: 1962
NO-RAG: 1880s
EVIDENCE FOUND: When they were first observed in 1962, some plant biologists dismissed the structures as artifactual, claiming that stromules were just oddly shaped chloroplasts with constricted regions or dividing chloroplasts
SELF-RAG FINAL ANSWER: 1962


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


QUESTION: Which artist who had a major influence on the Gothic Revival is represented in the V&A's British galleries?
GOLD: Horace Walpole
NO-RAG: george wilson
EVIDENCE FOUND: Patrons who have influenced taste are also represented by works of art from their collections, these include: Horace Walpole (a major influence on the Gothic Revival), William Thomas Beckford and Thomas Hope
SELF-RAG FINAL ANSWER: Horace Walpole


In [ ]:
# =========================
# DETAILED SINGLE EXAMPLE
# =========================

def show_detailed_example(example_id=0):
    ex = eval_ds[example_id]
    q = ex["question"]
    gold = ex["answers"]["text"][0]

    passages, scores = retrieve(q, top_k=5)

    print("=" * 100)
    print("QUESTION:", q)
    print("GOLD:", gold)
    print("=" * 100)

    for i, (p, s) in enumerate(zip(passages, scores), 1):
        print(f"\nPASSAGE {i} (score={s:.4f}):")
        print(p[:500])

    tokenizer, model = load_seq2seq(best_checkpoint)

    print("\n" + "="*100)

    pred_no = generate_text(tokenizer, model, build_prompt(q, None, "no_rag"))
    print("\nNO-RAG:\n", pred_no)

    pred_rag = generate_text(tokenizer, model, build_prompt(q, passages, "rag"))
    print("\nRAG:\n", pred_rag)

    pred_self, _ = self_rag_lite(tokenizer, model, q, passages)
    print("\nSELF-RAG:\n", pred_self)

    print("=" * 100)

    del model, tokenizer
    gc.collect()

# Run it
show_detailed_example(0)

QUESTION: In what year did Massachusetts first require children to be educated in schools?
GOLD: 1852

PASSAGE 1 (score=0.6324):
Private schooling in the United States has been debated by educators, lawmakers and parents, since the beginnings of compulsory education in Massachusetts in 1852. The Supreme Court precedent appears to favor educational choice, so long as states may set standards for educational accomplishment. Some of the most relevant Supreme Court case law on this is as follows: Runyon v. McCrary, 427 U.S. 160 (1976); Wisconsin v. Yoder, 406 U.S. 205 (1972); Pierce v. Society of Sisters, 268 U.S. 510 (1925);

PASSAGE 2 (score=0.3531):
In the 1890s, the University of Chicago, fearful that its vast resources would injure smaller schools by drawing away good students, affiliated with several regional colleges and universities: Des Moines College, Kalamazoo College, Butler University, and Stetson University. In 1896, the university affiliated with Shimer College in Mount Carr

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning




NO-RAG:
 1904

RAG:
 [3]

SELF-RAG:
 1852


In [ ]:
success_pool = method_results[
    (method_results['mode'] == 'self_rag_lite') & (method_results['f1'] > 0.8)
].merge(
    method_results[(method_results['mode'] == 'rag') & (method_results['f1'] < 0.5)],
    on='example_id'
)['example_id'].unique().tolist()
success_ids = success_pool[:7]

failure_pool = method_results[
    (method_results['mode'] == 'self_rag_lite') & (method_results['f1'] < 0.2)
]['example_id'].unique().tolist()
failure_ids = failure_pool[:3]

comparison_ids = []
max_len = max(len(success_ids), len(failure_ids))
for i in range(max_len):
    if i < len(success_ids): comparison_ids.append(success_ids[i])
    if i < len(failure_ids): comparison_ids.append(failure_ids[i])

pivot_df = method_results[method_results['example_id'].isin(comparison_ids)].pivot(
    index=['example_id', 'question', 'gold'],
    columns='mode',
    values='prediction'
).reset_index()

f1_scores = method_results[method_results['mode'] == 'self_rag_lite'][['example_id', 'f1', 'support']]
pivot_df = pivot_df.merge(f1_scores, on='example_id')

pivot_df['example_id'] = pd.Categorical(pivot_df['example_id'], categories=comparison_ids, ordered=True)
pivot_df = pivot_df.sort_values('example_id')

display(pivot_df[['question', 'gold', 'no_rag', 'rag', 'self_rag_lite', 'f1', 'support']])

,question,gold,no_rag,rag,self_rag_lite,f1,support
0,In what year did Massachusetts first require children to be educated in schools?,1852,1904,[3],1852,1.0,1.0
4,Who started rumors in 2008 that ABC would sell its ten owned-and-operated stations?,Caris & Co.,samuel wilson,[3],Not found,0.0,0.0
1,When were stromules discovered?,1962,1880s,unanswerable,1962,1.0,1.0
6,Why was this short termed organization created?,coordinate the response to the embargo,to provide a forum for eminent eminents,"national courts have a duty to interpret domestic law ""as far as possible in the light of",Not found,0.0,0.0
2,Who got a touchdown making the score 10-7?,Jonathan Stewart,fumble recoveries,[3],Jonathan Stewart,1.0,1.0
7,What does LGM stands for?,Last Glacial Maximum,LGM,HIV/AIDS,Not found,0.0,0.0
3,How many Examination Boards exist in India?,30,four,[1],30,1.0,0.0
5,Which network broadcasted the 50th Super Bowl game?,CBS,nba,[1],CBS,1.0,1.0
8,What studio does ABC own at 1500 Broadway in NYC?,Times Square Studios,symphony,renamed Peter Jennings Way,Times Square Studios,1.0,1.0
9,How many fraternities are apart of the university?,fifteen,ten,[3] There are fifteen fraternities,fifteen,1.0,0.0


# Results

In [ ]:
print("\n=== MODEL COMPARISON ===")
print(model_summary.sort_values("f1", ascending=False))

print("\n=== METHOD COMPARISON ===")
print(method_summary.sort_values("f1", ascending=False))

best_mode = method_summary.sort_values(["f1", "support"], ascending=False).iloc[0]["mode"]

print(f"\nBest method: {best_mode}")
print(f"Best model: {best_model_name}")


=== MODEL COMPARISON ===
                model    em        f1   support  retrieval_hit
1       Flan-T5-Small  0.10  0.187955  0.650000           0.85
0        Flan-T5-Base  0.05  0.148938  0.850000           0.85
2  LaMini-Flan-T5-77M  0.05  0.132890  0.901116           0.85

=== METHOD COMPARISON ===
            mode   em        f1  support
2  self_rag_lite  0.6  0.684524     0.75
1            rag  0.1  0.187955     0.65
0         no_rag  0.0  0.035000     0.10

Best method: self_rag_lite
Best model: Flan-T5-Small
